## EJERCICIO ENUNCIADO

Usaremos el dataset de madrid idealista:

https://www.kaggle.com/datasets/kanchana1990/madrid-idealista-property-listings


4 columnas numéricas

* price
* bedrooms
* bathrooms
* m2
* address (quitarle lo de ", Madrid" con numpy)

En cada apartado hacer visualizaciones con matplotlib o seaborn.

* 25 %:

* Carga de datos: cargarlo con np.genfromtext usar encoding="utf-8"
* Media, mediana
* Máximo y mínimo
* histograma y curva de densidad

* 25 %

* Cuartiles: Q1 (25), Q2 (50), Q3 (75)
* IQR
* Filtrar 20 % más caro, y el 20 % más barato
* Opcional: filtrar los barrios 20 % más baratos
* Moda: calcular moda también de address
* Opcional: Moda de los barrios más baratos y más caros
* Dispersión: varianza y desviación estándar

25 % 

* Filtro de outliers: tukey, z-score, marcar en un gráfico los límites de outliers: rojo y azul.
* Correlación: calcular la matriz y pintarla con matplotlib/seaborn
* Estandarización

25 % 

* Asimetría y curtosis
* Transformar distribuciones e interpretar resultados
* Contraste de hipótesis:
    * Que las casas de X barrio son más baratas de las de Y barrio
    * Que las casas de >= 3 baños son más caras que las casas de 1-2 baños


Entrega: 27/12


Datasets opcionales, alternativas para futuros módulos:

* https://www.kaggle.com/datasets/harlfoxem/housesalesprediction
* https://www.kaggle.com/datasets/desalegngeb/students-exam-scores
* https://www.kaggle.com/datasets/ziya07/music-education-performance-data

Fuentes datos:

* https://www.kaggle.com/datasets
* https://datos.gob.es/es/catalogo?res_format_label=CSV
* https://datasetsearch.research.google.com/
* seaborn.load_dataset('')
* https://github.com/mwaskom/seaborn-data

* Opcional
* MySQL Community Server + MySQL Workbench

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Carga de datos
file_path = "idealista_madrid.csv"  # Cambia esta ruta según corresponda
try:
    # Carga del dataset utilizando pandas (más robusto para CSVs complejos)
    df = pd.read_csv(file_path, encoding="utf-8")
except ValueError as e:
    print("Error al cargar los datos:", e)
    exit()

# Preprocesamiento inicial
# Quitar ", Madrid" del campo address
if 'address' in df.columns:
    df['address'] = df['address'].str.replace(", Madrid", "", regex=False)

# Renombrar columnas para mayor claridad (opcional)
df.rename(columns={
    'price': 'price',
    'rooms': 'bedrooms',
    'baths': 'bathrooms',
    'sqft': 'm2'
}, inplace=True)

# Convertir columnas relevantes a numérico
numeric_cols = ['price', 'bedrooms', 'bathrooms', 'm2']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Estadísticas iniciales
print("Media:")
print(df[numeric_cols].mean())
print("Mediana:")
print(df[numeric_cols].median())
print("Máximo y mínimo:")
print(df[numeric_cols].agg(['max', 'min']))

# Visualización inicial: Histograma y curva de densidad
for col in numeric_cols:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col].dropna(), kde=True, bins=30, color="blue")
    plt.title(f"Histograma y Curva de Densidad: {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

# Cuartiles e IQR
print("Cuartiles:")
print(df[numeric_cols].quantile([0.25, 0.5, 0.75]))
print("IQR:")
print(df[numeric_cols].apply(lambda x: stats.iqr(x.dropna())))

# Filtrar 20% más caro y 20% más barato
percentile_80 = df['price'].quantile(0.8)
percentile_20 = df['price'].quantile(0.2)

top_20 = df[df['price'] >= percentile_80]
bottom_20 = df[df['price'] <= percentile_20]

# Moda
print("Moda de address:")
print(df['address'].mode())

# Dispersión: Varianza y Desviación estándar
print("Varianza y Desviación Estándar:")
print(df[numeric_cols].var())
print(df[numeric_cols].std())

# Filtro de outliers: Tukey y Z-Score
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]

    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[col])
    plt.axvline(lower_bound, color='red', linestyle='--', label='Límite inferior')
    plt.axvline(upper_bound, color='blue', linestyle='--', label='Límite superior')
    plt.title(f"Detección de Outliers: {col}")
    plt.legend()
    plt.show()

# Correlación
corr_matrix = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de Correlación")
plt.show()

# Estandarización
df_standardized = df[numeric_cols].apply(lambda x: (x - x.mean()) / x.std())
print("Datos estandarizados (Primeras filas):")
print(df_standardized.head())

# Asimetría y Curtosis
for col in numeric_cols:
    skewness = stats.skew(df[col].dropna())
    kurtosis = stats.kurtosis(df[col].dropna())
    print(f"Asimetría ({col}): {skewness}")
    print(f"Curtosis ({col}): {kurtosis}")

# Contraste de hipótesis
# Casas de X barrio vs Y barrio
barrio_x = df[df['address'] == "Nombre Barrio X"]['price']
barrio_y = df[df['address'] == "Nombre Barrio Y"]['price']

stat, p_value = stats.ttest_ind(barrio_x, barrio_y, nan_policy='omit')
print(f"Contraste de hipótesis (Barrio X vs Barrio Y): p-value={p_value}")

# Casas >= 3 baños vs 1-2 baños
baths_3plus = df[df['bathrooms'] >= 3]['price']
baths_1_2 = df[(df['bathrooms'] == 1) | (df['bathrooms'] == 2)]['price']

stat, p_value = stats.ttest_ind(baths_3plus, baths_1_2, nan_policy='omit')
print(f"Contraste de hipótesis (>= 3 baños vs 1-2 baños): p-value={p_value}")
